# Text generation with an RNN
This notebook demonstrates how to generate text using an RNN. We will train a model to learn patterns in text, and then use that model to generate new text that is similar to the training data. For this demonstration we will consider the texts as sequence of characters, and we will train a model to predict the next character in a sequence.

# Setup and data preparation
The dataset we propose to use is an ebook of Alice in Wonderland, available at https://www.gutenberg.org/ebooks/11. The Gutenberg project provides a large collection of public domain books that can be used for free. You can try this experiment with any other book you like, but take care of the training times since LSTM models can be quite slow to train.

In [1]:
import tensorflow as tf
import numpy as np
import os
import time

In [2]:
name = "alice"
path_to_file = f'./{name}.txt'
nb_epochs = 60
model_filename = f"ps4_firstLSTM_{name}_{nb_epochs}epochs.keras"

raw_text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
n_chars = len(raw_text)
print(f'Length of text: {n_chars} characters')
print(raw_text[:250])


Length of text: 147955 characters
﻿Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rab


From this text we can build a dictionary of all unique characters. This will be the vocabulary that our model will use.

In [3]:
vocab = sorted(set(raw_text))
n_vocab = len(vocab)
print(f'{n_vocab} unique characters')

77 unique characters


## Vectorization of the text
Before training our model, we need to convert the texts to numerical representations. The most simple way is to map each unique character of our vocabulary to a numeric ID. To do so, we built two dictionaries: one that maps characters to IDs, and another that maps IDs to characters.

In [4]:
char_to_int = dict((c, i) for i, c in enumerate(vocab))
int_to_char = dict((i, c) for i, c in enumerate(vocab))

sample_chars = raw_text[:14]
sample_ids = [char_to_int[char] for char in sample_chars]
print(sample_chars)
print(sample_ids)

﻿Alice’s Adven
[76, 15, 55, 52, 46, 48, 73, 62, 2, 15, 47, 65, 48, 57]


In [5]:
sample_chars_again = [int_to_char[i] for i in sample_ids]
print(sample_chars_again)

['\ufeff', 'A', 'l', 'i', 'c', 'e', '’', 's', ' ', 'A', 'd', 'v', 'e', 'n']


# Prepare the training set
We now need to define the training data. There is a lot of flexibility in how you choose to break up the text and expose it to the RNN during training. Here, we will split the text into sequences with a fixed length of 100 characters. As an alternative, one could split the data by sentences, padding the shorter sequences and truncating the longer ones.

Each training pattern of the network comprises 100 time steps of one character (X) followed by one character output (y). When creating these sequences, we slide this window along the whole text one character at a time, allowing each character a chance to be learned from the 100 characters that preceded it (except the first 100 characters, of course).

For example, if the sequence length is 3 (for simplicity), then the first two training patterns would be as follows:
- Sequence 1: "Fir" -> "s"
- Sequence 2: "irs" -> "t"

As we split the text into these sequences, we convert the characters to integers using the lookup table we prepared earlier.

In [6]:
seq_length = 100
dataX = []
dataY = []
for i in range(0, n_chars - seq_length, 1):
	seq_in = raw_text[i:i + seq_length]
	seq_out = raw_text[i + seq_length]
	dataX.append([char_to_int[char] for char in seq_in])
	dataY.append(char_to_int[seq_out])
n_sequences = len(dataX)
print("Total Patterns: ", n_sequences)


Total Patterns:  147855


This is our training set. To feed it to a Keras LSTM model, we need 
- to convert the intputs into 3D arrays of shape `(n_sequences, seq_length, 1)`, and the outputs into 2D arrays of shape `(n_sequences, n_vocab)`
- to rescale the integers to the range 0-to-1, as usual with neural networks
- to one-hot encode the outputs to allow the model to predict the probability of each character in the vocabulary, instead of predicting the character directly.

In [7]:
# reshape X to be [samples, time steps, features]
X = np.reshape(dataX, (n_sequences, seq_length, 1))
# normalize
X = X / float(n_vocab)
# one hot encode the output variable
y = tf.keras.utils.to_categorical(dataY)

# Build the model
Let's now define the model. We will use a single hidden LSTM layer with 256 memory units. The output layer will be a dense layer with a softmax activation function, to predict the probabilities of each character in the vocabulary.

The text generation problem is actually a single character classification problem with `n_vocab` classes and, as such, is defined as optimizing the log loss (cross entropy) using the ADAM optimization algorithm for speed.

In [8]:
# define the LSTM model
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.LSTM(256, input_shape=(X.shape[1], X.shape[2])))
model.add(tf.keras.layers.Dropout(0.2))
model.add(tf.keras.layers.Dense(y.shape[1], activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam')

callbacks = [tf.keras.callbacks.ModelCheckpoint(model_filename, monitor='loss', verbose=1, save_best_only=True, mode='min')]
model.fit(X, y, epochs=nb_epochs, batch_size=128, callbacks=callbacks)

Epoch 1/60


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1156/1156 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step - loss: 3.2206
Epoch 1: loss improved from inf to 3.10680, saving model to ps4_firstLSTM_alice_60epochs.keras
1156/1156 ━━━━━━━━━━━━━━━━━━━━ 249s 215ms/step - loss: 3.2205
Epoch 2/60
1156/1156 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step - loss: 2.9338
Epoch 2: loss improved from 3.10680 to 2.90662, saving model to ps4_firstLSTM_alice_60epochs.keras
1156/1156 ━━━━━━━━━━━━━━━━━━━━ 291s 251ms/step - loss: 2.9337
Epoch 3/60
1156/1156 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step - loss: 2.8287
Epoch 3: loss improved from 2.90662 to 2.80068, saving model to ps4_firstLSTM_alice_60epochs.keras
1156/1156 ━━━━━━━━━━━━━━━━━━━━ 274s 237ms/step - loss: 2.8287
Epoch 4/60
1156/1156 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - loss: 2.7240
Epoch 4: loss improved from 2.80068 to 2.71117, saving model to ps4_firstLSTM_alice_60epochs.keras
1156/1156 ━━━━━━━━━━━━━━━━━━━━ 287s 248ms/step - loss: 2.7240
Epoch 5/60
1156/1156 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step - loss: 2.6567
Epoch 5: loss im

# Test the model
The simplest way to use the LSTM model to make predictions is to first start with a seed sequence as input, generate the next character, then update the seed sequence to add the generated character on the end and trim off the first character. This process is repeated for as long as you want to predict new characters (e.g., a sequence of 1,000 characters in length).

In [13]:
import sys
import tensorflow as tf
import numpy as np
import os
import time
name = "alice"
path_to_file = f'./{name}.txt'
model_filename = f"ps4_firstLSTM_{name}_40epochs.keras"

raw_text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
n_chars = len(raw_text)
vocab = sorted(set(raw_text))
n_vocab = len(vocab)
char_to_int = dict((c, i) for i, c in enumerate(vocab))
int_to_char = dict((i, c) for i, c in enumerate(vocab))
seq_length = 100
dataX = []
dataY = []
for i in range(0, n_chars - seq_length, 1):
	seq_in = raw_text[i:i + seq_length]
	seq_out = raw_text[i + seq_length]
	dataX.append([char_to_int[char] for char in seq_in])
	dataY.append(char_to_int[seq_out])
n_sequences = len(dataX)
print("Total Patterns: ", n_sequences)

model = tf.keras.models.load_model(model_filename)

# pick a random seed
start = np.random.randint(0, len(dataX)-1)
pattern = dataX[start]
print("Seed:")
print("\"", ''.join([int_to_char[value] for value in pattern]), "\"")
# generate characters
for i in range(1000):
	x = np.reshape(pattern, (1, len(pattern), 1)) # reshape to 3D array
	x = x / float(n_vocab) # normalize
	prediction = model.predict(x, verbose=0) # predict the next character
	index = np.argmax(prediction) # get the index of the character with the highest probability
	result = int_to_char[index] # get the corresponding character
	seq_in = [int_to_char[value] for value in pattern] # get the input sequence
	sys.stdout.write(result) # print the character (no newline)
	pattern.append(index) # update the input sequence by adding the predicted character
	pattern = pattern[1:len(pattern)] # ...and by removing the first character
print("\nDone.")

Total Patterns:  147855
Seed:
" ce, surprised at her own courage. “It’s
no business of _mine_.”

The Queen turned crimson with fu "
rouus to her aeain, and shen she had
aot toted an the could, and shened th the terter of the tohee oe the
court, and the test hn woshd the whsl oo tee shene wosld the tan oo
the whole was in ohc foore te the soher. “hh whu wou geeen to bean to
teee it thi tenteng  and the shene wo be a geld touhd the wan noto the
court.

“Whet aor toee thet,” said the Daterpillar.

“Ie dourse tou coow aedun ie,” said the Katter. “I wond the toond tele
te the boeste of the tomet is the wonle.”

“I soihk to aelin in ih eia,” Alice replied tety porily. “Ieee io ha
tou  aod you make th tee then ”hu, I whnll— ”hu Inneh soone oo  anditt
_oe tound to tee thet io the wey,”

“I dou’d thel ano ”hur sheug,” said the Kanthr, “io I san to mere the
coust wo tha thmtsee.”

“I don’t keke the woulo,” said the Kanter, “io I san to mep aelig if
tou fonn thet ”ou’h note to bearn ”

“I don’d t